1. Import Libraries and Load Data

In [9]:
import numpy as np
import pandas as pd

# Load the dataset
df = pd.read_csv("Messy_data.csv")
print("Dataset loaded successfully!")

Dataset loaded successfully!


2. Initial Data Inspection (head, tail, info, describe, nulls)

In [10]:
print("--- FIRST 5 ROWS (head) ---")
display(df.head())

print("\n--- LAST 5 ROWS (tail) ---")
display(df.tail())

print("\n--- DATASET SHAPE ---")
print(f"Total Rows: {df.shape[0]}, Total Columns: {df.shape[1]}")

print("\n--- DATA TYPES & NON-NULL SUMMARY (info) ---")
df.info()

print("\n--- STATISTICAL SUMMARY (describe) ---")
display(df.describe(include="all"))

print("\n--- MISSING VALUES PER COLUMN ---")
print(df.isnull().sum())

print("\n--- DUPLICATE ROWS COUNT ---")
print(f"Duplicates: {df.duplicated().sum()}")

--- FIRST 5 ROWS (head) ---


,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge
0,10001,Nikhil Jain,Mumbai,Laptop,3.0,50790.72,2026-01-18,Completed,61.0
1,10002,Simran Kaur,delhi,Monitor,1.0,10307.27,2026-01-28,Pending,50.0
2,10003,Zoya Ali,Delhi,Headphones,5.0,2439.64,2026-02-27,Delivered,35.0
3,10004,Priya Bansal,Delhi,Tablet,4.0,20945.65,02/03/2026,Pending,66.0
4,10005,NaN,Mumbai,Phone,4.0,15822.27,2026-02-14,Delivered,34.0



--- LAST 5 ROWS (tail) ---


,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge
180,10006,Priya Bansal,Delhi,Smartwatch,5.0,5768.41,2026-02-18,Completed,53.0
181,10026,Kabir Singh,Delhi,Keyboard,1.0,1506.37,15-02-2026,Cancelled,31.0
182,10081,Aarav Sharma,Bengaluru,Headphones,1.0,2591.0,2026-01-31,Pending,48.0
183,10121,Neha Gupta,Pune,Smartwatch,3.0,6517.3,2026-02-04,Cancelled,48.0
184,10151,Isha Roy,Mumbai,Smartwatch,3.0,6791.02,2026-01-15,Pending,20.0



--- DATASET SHAPE ---
Total Rows: 185, Total Columns: 9

--- DATA TYPES & NON-NULL SUMMARY (info) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 185 entries, 0 to 184
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   OrderID       185 non-null    int64 
 1   CustomerName  183 non-null    object
 2   City          183 non-null    object
 3   Product       185 non-null    object
 4   Quantity      183 non-null    object
 5   UnitPrice     183 non-null    object
 6   OrderDate     183 non-null    object
 7   OrderStatus   183 non-null    object
 8   CustomerAge   183 non-null    object
dtypes: int64(1), object(8)
memory usage: 13.1+ KB

--- STATISTICAL SUMMARY (describe) ---


,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge
count,185.000000,183,183,185,183,183,183,183,183
unique,NaN,33,12,10,9,178,83,8,54
top,NaN,Simran Kaur,Bengaluru,Phone,1.0,5768.41,2026-03-31,Delivered,24.0
freq,NaN,12,33,34,42,2,7,42,8
mean,10090.135135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,52.229719,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,10001.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,10045.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,10090.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,10135.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



--- MISSING VALUES PER COLUMN ---
OrderID         0
CustomerName    2
City            2
Product         0
Quantity        2
UnitPrice       2
OrderDate       2
OrderStatus     2
CustomerAge     2
dtype: int64

--- DUPLICATE ROWS COUNT ---
Duplicates: 5


3. Remove Duplicate Records

In [11]:
# Check and remove duplicates
initial_rows = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {initial_rows - len(df)} duplicate rows. Current rows: {len(df)}")

Removed 5 duplicate rows. Current rows: 180


4. Fix Whitespace and Standardize Text

In [12]:
# Strip extra leading/trailing whitespace across all text columns
text_cols = ["CustomerName", "City", "Product", "OrderStatus"]
for col in text_cols:
  df[col] = df[col].astype(str).str.strip().replace("nan", np.nan)

# Standardize City names to Title Case (e.g., 'delhi', 'DELHI' -> 'Delhi')
df["City"] = df["City"].str.title()

# Standardize OrderStatus to Title Case (e.g., 'completed ', 'SHIPPED' -> 'Completed', 'Shipped')
df["OrderStatus"] = df["OrderStatus"].str.title()

print("Unique Cities:", df["City"].unique())
print("Unique Order Statuses:", df["OrderStatus"].unique())

Unique Cities: ['Mumbai' 'Delhi' 'Chennai' 'Bengaluru' 'Pune' 'Kolkata' 'Gurugram'
 'Noida' nan]
Unique Order Statuses: ['Completed' 'Pending' 'Delivered' 'Shipped' 'Cancelled' nan]


5. Fix Data Type Errors & Conversions

In [13]:
# 1. Clean Quantity: replace word numbers & units, convert to float
quantity_map = {"three": "3", "2 pcs": "2"}
df["Quantity"] = df["Quantity"].replace(quantity_map)
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")

# 2. Clean UnitPrice: remove commas, handle 'unknown', convert to float
df["UnitPrice"] = df["UnitPrice"].astype(str).str.replace(",", "").str.strip()
df["UnitPrice"] = df["UnitPrice"].replace(["unknown", "nan"], np.nan)
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")

# 3. Clean CustomerAge: replace words/suffixes ('forty', '29 years'), convert to float
age_map = {"forty": "40", "29 years": "29"}
df["CustomerAge"] = df["CustomerAge"].replace(age_map)
df["CustomerAge"] = pd.to_numeric(df["CustomerAge"], errors="coerce")

# 4. Standardize OrderDate into datetime (coercing invalid entries like 'not a date')
df["OrderDate"] = pd.to_datetime(df["OrderDate"], format="mixed", errors="coerce")

print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
OrderID                  int64
CustomerName            object
City                    object
Product                 object
Quantity               float64
UnitPrice              float64
OrderDate       datetime64[ns]
OrderStatus             object
CustomerAge            float64
dtype: object


6. Handle Outliers and Missing Values

In [14]:
# --- Outlier Handling ---
# Quantities <= 0 or extreme bulk errors (> 50) set to NaN
df.loc[(df["Quantity"] <= 0) | (df["Quantity"] > 50), "Quantity"] = np.nan

# Unrealistic Customer Ages (< 10 or > 100) set to NaN
df.loc[(df["CustomerAge"] < 10) | (df["CustomerAge"] > 100), "CustomerAge"] = (
    np.nan
)

# Unrealistic UnitPrices (< 0 or extreme) set to NaN
df.loc[df["UnitPrice"] <= 0, "UnitPrice"] = np.nan

# --- Missing Value Imputation ---
# Impute numerical columns with Median (robust to outliers)
df["Quantity"] = df["Quantity"].fillna(df["Quantity"].median())
df["UnitPrice"] = df["UnitPrice"].fillna(
    df.groupby("Product")["UnitPrice"].transform("median")
)
df["CustomerAge"] = df["CustomerAge"].fillna(
    round(df["CustomerAge"].median())
).astype(int)

# Impute categorical/text columns with Mode or descriptive placeholder
df["CustomerName"] = df["CustomerName"].fillna("Unknown Customer")
df["City"] = df["City"].fillna(df["City"].mode()[0])
df["OrderStatus"] = df["OrderStatus"].fillna("Pending")

# Forward fill or use median date for missing/invalid OrderDate
df["OrderDate"] = df["OrderDate"].ffill()

print("Remaining null values:")
print(df.isnull().sum())

Remaining null values:
OrderID         0
CustomerName    0
City            0
Product         0
Quantity        0
UnitPrice       0
OrderDate       0
OrderStatus     0
CustomerAge     0
dtype: int64


7. Feature Engineering (TotalAmount) and Final Export

In [15]:
# Create TotalAmount column (Quantity * UnitPrice rounded to 2 decimals)
df["TotalAmount"] = (df["Quantity"] * df["UnitPrice"]).round(2)

# Final inspection
print("--- FINAL INSPECTION ---")
display(df.head(10))
print(df.info())

# Export the cleaned dataset
df.to_csv("cleaned_sales_dataset.csv", index=False)
print("File exported successfully as 'cleaned_sales_dataset.csv'!")

--- FINAL INSPECTION ---


,OrderID,CustomerName,City,Product,Quantity,UnitPrice,OrderDate,OrderStatus,CustomerAge,TotalAmount
0,10001,Nikhil Jain,Mumbai,Laptop,3.0,50790.72,2026-01-18,Completed,61,152372.16
1,10002,Simran Kaur,Delhi,Monitor,1.0,10307.27,2026-01-28,Pending,50,10307.27
2,10003,Zoya Ali,Delhi,Headphones,5.0,2439.64,2026-02-27,Delivered,35,12198.20
3,10004,Priya Bansal,Delhi,Tablet,4.0,20945.65,2026-02-03,Pending,66,83782.60
4,10005,Unknown Customer,Mumbai,Phone,4.0,15822.27,2026-02-14,Delivered,34,63289.08
5,10006,Priya Bansal,Delhi,Smartwatch,5.0,5768.41,2026-02-18,Completed,53,28842.05
6,10007,Meera Joshi,Chennai,Headphones,1.0,2159.37,2026-01-30,Shipped,23,2159.37
7,10008,Sara Thomas,Bengaluru,Phone,3.0,16801.06,2026-03-23,Shipped,28,50403.18
8,10009,Tara Kapoor,Mumbai,Headphones,3.0,2651.37,2026-03-29,Completed,56,7954.11
9,10010,Nikhil Jain,Pune,Headphones,2.0,2471.70,2026-02-04,Delivered,32,4943.40


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   OrderID       180 non-null    int64         
 1   CustomerName  180 non-null    object        
 2   City          180 non-null    object        
 3   Product       180 non-null    object        
 4   Quantity      180 non-null    float64       
 5   UnitPrice     180 non-null    float64       
 6   OrderDate     180 non-null    datetime64[ns]
 7   OrderStatus   180 non-null    object        
 8   CustomerAge   180 non-null    int64         
 9   TotalAmount   180 non-null    float64       
dtypes: datetime64[ns](1), float64(3), int64(2), object(4)
memory usage: 14.2+ KB
None
File exported successfully as 'cleaned_sales_dataset.csv'!
